# Audio Recording & Acoustic Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensein/senselab/blob/main/tutorials/audio/audio_recording_and_acoustic_analysis.ipynb)

In this tutorial you will learn how to:

1. **Load audio** into senselab's `Audio` data structure
2. **Visualize waveforms and spectrograms** to see what sound "looks like"
3. **Extract pitch (F0)** to measure how high or low a voice sounds
4. **Extract formants (F1, F2)** to characterize vowel quality and vocal tract shape
5. **Detect emotion** from speech using a pre-trained model
6. **Compare two speakers** to determine whether audio samples come from the same person

These are foundational skills for voice and speech analysis in behavioral research.

In [ ]:
# Install senselab and dependencies
!pip install -q uv
!uv pip install --pre --system "senselab"

> **\u26a0\ufe0f Restart runtime after install**
>
> The install may upgrade packages (e.g., numpy) that are already loaded.
> Go to **Runtime \u2192 Restart session**, then **run all cells below** (skip this install cell).

In [ ]:
import os

import matplotlib.pyplot as plt
import torch

from senselab.audio.data_structures import Audio
from senselab.audio.tasks.classification.speech_emotion_recognition.api import (
    classify_emotions_from_speech,
)
from senselab.audio.tasks.features_extraction.praat_parselmouth import (
    extract_praat_parselmouth_features_from_audios,
)
from senselab.audio.tasks.features_extraction.torchaudio import (
    extract_pitch_from_audios,
)
from senselab.audio.tasks.plotting.plotting import (
    play_audio,
    plot_specgram,
    plot_waveform,
)
from senselab.audio.tasks.preprocessing.preprocessing import (
    downmix_audios_to_mono,
    resample_audios,
)
from senselab.audio.tasks.speaker_verification.speaker_verification import (
    verify_speaker,
)
from senselab.utils.data_structures import DeviceType, HFModel, SpeechBrainModel

# Auto-detect device
device = DeviceType.CUDA if torch.cuda.is_available() else DeviceType.CPU
print(f"Using device: {device.value}")

## 1. Record or Load Audio

Audio is a digital representation of sound waves. Key concepts:

- **Waveform**: a sequence of amplitude values sampled at regular intervals. Each value
  represents the air pressure displacement at that instant.
- **Sampling rate**: how many samples are captured per second. For example, 16 000 Hz
  means 16 000 amplitude values per second of audio.
- **Channels**: mono (1 channel) captures a single signal; stereo (2 channels) captures
  left and right signals separately.

Below we download a sample speech file from the senselab test data. In a live
session you could record your own audio instead.

In [ ]:
# Download a sample audio file for analysis
import urllib.request
from pathlib import Path

Path("tutorial_audio_files").mkdir(exist_ok=True)

base_url = "https://github.com/sensein/senselab/raw/main/src/tests/data_for_testing"
for fname in ["audio_48khz_mono_16bits.wav", "audio_48khz_stereo_16bits.wav"]:
    dest = Path("tutorial_audio_files") / fname
    if not dest.exists():
        urllib.request.urlretrieve(f"{base_url}/{fname}", str(dest))

audio = Audio(filepath=os.path.abspath("tutorial_audio_files/audio_48khz_mono_16bits.wav"))
print(f"Loaded audio: {audio.waveform.shape[1]} samples at {audio.sampling_rate} Hz")
print(f"Duration: {audio.waveform.shape[1] / audio.sampling_rate:.2f} seconds")
print(f"Channels: {audio.waveform.shape[0]}")

## 2. Waveform Visualization

A waveform plot shows amplitude (loudness) over time. Positive and negative values
represent the air pressure changes that make up sound. Louder sounds have larger
amplitude swings; quiet passages stay close to zero.

In [ ]:
plot_waveform(audio)
plt.show()

play_audio(audio)

## 3. Spectrogram

A spectrogram shows how the frequency content of audio changes over time. Think of
it as a "heat map" where:

- **X-axis** = time
- **Y-axis** = frequency
- **Color** = intensity (energy at that frequency and time)

The **mel scale** is a perceptual frequency scale that better matches how humans
hear -- lower frequencies are spaced more widely, reflecting the fact that we are
more sensitive to differences at low frequencies than at high ones.

In [ ]:
# Linear frequency spectrogram
plot_specgram(audio, title="Linear Spectrogram")
plt.show()

# Mel-scale spectrogram (better matches human hearing)
plot_specgram(audio, mel_scale=True, title="Mel Spectrogram")
plt.show()

## 4. Pitch (F0) Extraction

**Pitch** (fundamental frequency, F0) is the rate at which your vocal folds vibrate.
It determines how "high" or "low" your voice sounds.

- Typical range: ~80 Hz (deep voice) to ~300 Hz (high voice)
- **Pitch contour** -- how pitch changes over time -- carries information about
  intonation, stress, and emotion.

Below we resample to 16 kHz (a standard rate for many speech models) and extract
the pitch contour using normalized cross-correlation.

In [ ]:
# Resample to 16 kHz for pitch extraction
audio_16k = resample_audios([audio], resample_rate=16000)[0]

# Extract pitch contour
pitch_result = extract_pitch_from_audios([audio_16k], freq_low=80, freq_high=500)
pitch_contour = pitch_result[0]["pitch"].squeeze()

# Build a time axis (default hop length for torchaudio pitch detection)
hop_length = 160  # default for 16 kHz
time_axis = torch.arange(len(pitch_contour)) * hop_length / 16000

# Plot only voiced frames (pitch > 0)
plt.figure(figsize=(12, 4))
voiced = pitch_contour > 0
plt.scatter(time_axis[voiced], pitch_contour[voiced], s=2, c="blue")
plt.xlabel("Time (seconds)")
plt.ylabel("Frequency (Hz)")
plt.title("Pitch (F0) Contour")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Formant Extraction

**Formants** are resonant frequencies of the vocal tract -- the natural
amplification that occurs at certain frequencies due to the shape of your mouth,
throat, and nasal cavities.

- **F1** (first formant): correlates with tongue height. Low F1 = high tongue
  (as in "ee"); high F1 = low tongue (as in "ah").
- **F2** (second formant): correlates with tongue front/back position. High F2 =
  front vowel ("ee"); low F2 = back vowel ("oo").

Together, F1 and F2 can distinguish most vowels -- this is why a vowel space plot
(F1 vs F2) is a standard tool in phonetics.

In [ ]:
# Extract time-varying formant tracks using Praat/Parselmouth
import parselmouth
import numpy as np

snd = parselmouth.Sound(audio.waveform.squeeze().numpy(), sampling_frequency=audio.sampling_rate)
formant = snd.to_formant_burg()

# Get F1 and F2 at each time frame
times = formant.xs()
f1_values = np.array([formant.get_value_at_time(1, t) for t in times])
f2_values = np.array([formant.get_value_at_time(2, t) for t in times])

# Plot formant tracks over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# F1 track
valid_f1 = ~np.isnan(f1_values)
ax1.scatter(np.array(times)[valid_f1], f1_values[valid_f1], s=1, c="blue", alpha=0.6)
ax1.set_ylabel("F1 Frequency (Hz)")
ax1.set_title("Formant Tracks Over Time")
ax1.grid(True, alpha=0.3)

# F2 track
valid_f2 = ~np.isnan(f2_values)
ax2.scatter(np.array(times)[valid_f2], f2_values[valid_f2], s=1, c="red", alpha=0.6)
ax2.set_ylabel("F2 Frequency (Hz)")
ax2.set_xlabel("Time (seconds)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Also print summary statistics
print("=== Formant Summary ===")
print(f"  F1 mean: {np.nanmean(f1_values):.0f} Hz (std: {np.nanstd(f1_values):.0f} Hz)")
print(f"  F2 mean: {np.nanmean(f2_values):.0f} Hz (std: {np.nanstd(f2_values):.0f} Hz)")

## 6. Speech Emotion Recognition

Speech emotion recognition (SER) uses machine learning to detect the emotional
state of a speaker from acoustic cues -- not the words themselves, but **how**
they are spoken. Features like pitch variability, speech rate, energy patterns,
and spectral characteristics all carry emotional information.

We use a pre-trained wav2vec2-based model fine-tuned for emotion classification.

In [ ]:
# Classify emotion from speech
model = HFModel(path_or_uri="ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition")
results = classify_emotions_from_speech([audio_16k], model=model, device=device)

print("=== Emotion Classification ===")
for label, score in zip(results[0].labels, results[0].scores):
    bar = "\u2588" * int(score * 40)
    print(f"  {label:12s}: {score:.3f} {bar}")

## 7. Speaker Matching

Speaker verification determines whether two audio samples come from the same
person. It works by:

1. Extracting a **speaker embedding** -- a compact numerical representation of
   voice characteristics (timbre, pitch range, speaking style).
2. Comparing embeddings using **cosine similarity**.
3. If similarity exceeds a threshold, the speakers are likely the same person.

Below we load a second audio file (stereo), downmix it to mono, resample to
16 kHz, and compare the two speakers.

In [ ]:
# Load the second audio file (already downloaded above)
audio2 = Audio(filepath=os.path.abspath("tutorial_audio_files/audio_48khz_stereo_16bits.wav"))

# Downmix stereo to mono, then resample to 16 kHz
audio2_mono = downmix_audios_to_mono([audio2])[0]
audio2_16k = resample_audios([audio2_mono], resample_rate=16000)[0]

# Compare the two speakers
similarity_score, is_same_speaker = verify_speaker([(audio_16k, audio2_16k)])[0]

print("=== Speaker Verification ===")
print(f"  Similarity score: {similarity_score:.4f}")
print(f"  Same speaker: {'Yes' if is_same_speaker else 'No'}")
print()
if is_same_speaker:
    print("  The model believes these are from the SAME speaker.")
else:
    print("  The model believes these are from DIFFERENT speakers.")
print("  (Threshold: 0.25 -- scores above this indicate same speaker)")

## Summary

In this tutorial we covered:

- **Loading audio** into senselab's `Audio` object and inspecting its properties
  (sampling rate, duration, channels).
- **Waveform visualization** to see amplitude over time, and **spectrograms**
  (linear and mel-scale) to see frequency content over time.
- **Pitch (F0) extraction** to measure vocal fold vibration rate and plot the
  intonation contour.
- **Formant extraction** (F1, F2) using Praat/Parselmouth to characterize vowel
  quality and vocal tract resonance.
- **Speech emotion recognition** using a pre-trained wav2vec2 model to classify
  the emotional tone of speech.
- **Speaker verification** to determine whether two recordings come from the
  same person using speaker embeddings and cosine similarity.

Next up: **Tutorial 2 -- Transcription & Phoneme Analysis**, where we will
transcribe speech to text, perform forced alignment, and analyze phoneme-level
features.